In [ ]:
!pip install langchain langchain-community pymupdf pypdf chromadb langchain-groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/1

In [ ]:
from langchain_core.documents import Document

doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Chennai Institute of Technology",
        "date_created":"2026-05-12"
    }
)
doc

Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'Chennai Institute of Technology', 'date_created': '2026-05-12'}, page_content='this is the main text content I am using to create RAG')

**After running next cell, upload 1 or more pdf in data folder**

In [ ]:
import os

os.makedirs("data/pdf_files",exist_ok=True)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

pdf_dir = Path("data/pdf_files")
# Find all PDF files recursively
pdf_files = list(pdf_dir.glob("**/*.pdf"))
print(f"Found {len(pdf_files)} PDF files to process")

Found 1 PDF files to process


In [ ]:
all_documents = []
for pdf_file in pdf_files:
    print(f"\nProcessing: {pdf_file.name}")
    try:
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()

        # Add source information to metadata
        for doc in documents:
            doc.metadata['source_file'] = pdf_file.name
            doc.metadata['file_type'] = 'pdf'

        all_documents.extend(documents)
        print(f"  ✓ Loaded {len(documents)} pages")
    except Exception as e:
        print(f"  ✗ Error: {e}")
print('Total number of pages: ', len(all_documents))


Processing: Project.pdf
  ✓ Loaded 7 pages
Total number of pages:  7


In [ ]:
all_documents[0]

Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-05-15T09:33:17+05:30', 'author': '', 'moddate': '2026-05-15T09:33:17+05:30', 'title': 'Microsoft Word - Project - I', 'source': 'data/pdf_files/Project.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'Project.pdf', 'file_type': 'pdf'}, page_content='Problem Statements \n1. Carpool & Shared Mobility System \nProblem Statement \nSingle-passenger vehicle usage increases fuel costs and traffic congestion. This project develops \na carpooling platform that allows users to share rides based on destination similarity. \nConstraints \n1. A user can register as either a Driver or a Passenger.  \n2. Each driver can offer only one active ride at a time.  \n3. A vehicle has a limited seating capacity (1–6 passengers).  \n4. Passengers can join rides only if seats are available.  \n5. Pickup and destination locations must be valid strings or predefined city names.  \n6. Ride matc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size=500
chunk_overlap=100

# Split documents into smaller chunks for better RAG performance

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(all_documents)
print(f"Split {len(all_documents)} documents into {len(chunks)} chunks")

Split 7 documents into 20 chunks


In [ ]:
print(f"\nExample chunk:")
print(f"\nContent: {chunks[0].page_content[:50]}...")
print(f"\nMetadata: {chunks[0].metadata}")


Example chunk:

Content: Problem Statements 
1. Carpool & Shared Mobility S...

Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-05-15T09:33:17+05:30', 'author': '', 'moddate': '2026-05-15T09:33:17+05:30', 'title': 'Microsoft Word - Project - I', 'source': 'data/pdf_files/Project.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'Project.pdf', 'file_type': 'pdf'}


In [ ]:
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [ ]:
## initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_4297/1809734092.py:24: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [ ]:
from typing import Any
import chromadb
import uuid

class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [ ]:
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
### Convert the text to embeddings
texts=[chunk.page_content for chunk in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 20 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (20, 384)
Adding 20 documents to vector store...
Successfully added 20 documents to vector store
Total documents in collection: 20


In [ ]:
from typing import Dict

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 3, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    # if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': doc_id,
                        'content': document,
                        'metadata': metadata,
                        'similarity_score': similarity_score,
                        'distance': distance,
                        'rank': i + 1
                    })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

In [ ]:
rag_retriever=RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [ ]:
rag_retriever.retrieve("what is molecular unfolding",2)

Retrieving documents for query: 'what is molecular unfolding'
Top K: 2, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_11a173e2_6',
  'content': 'C++ Concepts Used \n\uf0b7 Classes & Objects  \n\uf0b7 Inheritance  \n\uf0b7 Polymorphism  \n\uf0b7 STL (Vectors, Maps)  \n\uf0b7 File Handling',
  'metadata': {'producer': 'Microsoft: Print To PDF',
   'moddate': '2026-05-15T09:33:17+05:30',
   'content_length': 113,
   'source': 'data/pdf_files/Project.pdf',
   'file_type': 'pdf',
   'creationdate': '2026-05-15T09:33:17+05:30',
   'doc_index': 6,
   'creator': 'PyPDF',
   'source_file': 'Project.pdf',
   'page_label': '2',
   'author': '',
   'total_pages': 7,
   'page': 1,
   'title': 'Microsoft Word - Project - I'},
  'similarity_score': -0.7094894647598267,
  'distance': 1.7094894647598267,
  'rank': 1},
 {'id': 'doc_f97dc7f7_13',
  'content': '\uf0b7 STL (Vectors, Maps, Queues)  \n\uf0b7 File Handling  \n\uf0b7 Graph Algorithms (Shortest Path)  \n\uf0b7 Searching & Sorting  \n\uf0b7 Exception Handling',
  'metadata': {'source_file': 'Project.pdf',
   'source': 'data/pdf_files/Project.pdf',


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-120b", api_key: str =None):
        """
        Initialize Groq LLM

        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_rag_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated response
        """
        simple_prompt = f"""You are a helpful AI assistant. Answer the question accurately and concisely.

        Question: {query}

        Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

**add groq key  -- with name 'GROQ_API_KEY' in left side**

In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
from google.colab import userdata
groq_key = userdata.get('GROQ_API_KEY')

try:
    groq_llm = GroqLLM(api_key=groq_key)
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-120b
Groq LLM initialized successfully!


In [ ]:
groq_llm.generate_response_simple("Hi how are you?")

"I'm doing well, thank you! How can I assist you today?"

In [ ]:
groq_llm.generate_response_simple("Who is the CM of Tamil Nadu?")

'As of my latest information (June\u202f2024), the Chief Minister of Tamil\u202fNadu is **M.\u202fK.\u202fStalin**. I don’t have data beyond that date.'

In [ ]:
### Simple RAG pipeline with Groq LLM
def rag_pipeline(query, retriever, llm, top_k):
  results = retriever.retrieve(query, top_k)
  context = "\n\n".join([doc['content'] for doc in results]) if results else ""
  if context == "":
    return "No relevant context found to answer the question."
  response = llm.generate_rag_response(query, context)
  return response

In [ ]:
answer=rag_pipeline("how many projects and classes",rag_retriever,groq_llm, top_k=10)
print(answer)

Retrieving documents for query: 'how many projects and classes'
Top K: 10, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 10 documents (after filtering)
**Projects**

The material describes **two distinct software projects**:

1. **Carpool & Shared‑Mobility System** – a ride‑sharing platform that matches drivers and passengers, manages seats, routes, fare calculation, live tracking, etc.  
2. **Public Transport Management System** – a smart system for buses, metro trains, shuttles (and other modes) that handles registration, booking, scheduling, seat‑availability checks, fare estimation, live vehicle tracking, admin controls, reporting, etc.

**Classes**

The context lists the C++ concepts that should be used (classes & objects, inheritance, polymorphism, STL containers, etc.) and outlines many functional entities (users, drivers, passengers, vehicles, routes, bookings, tickets, admin dashboard, reports, etc.), but it never states an exact count of classes that must be implemented.

Therefore:

- **Number of projects:** **2**  
- **Number of classes:** N